# SMILES 2026 Method 3 Colab Runner

This notebook updates the GitHub repo in a Drive-backed workspace, installs dependencies, and runs the LLM-Check **attention-score-only** adaptation on `dataset.csv` with `Qwen/Qwen2.5-0.5B`.

Default Method 3 settings:

- attention score only
- response-only token span
- eager attention for Qwen
- repo-aligned skip of the first attention layer
- single combined JSON output across the 2 requested classifier configurations:
  - `logistic_regression`
  - `mlp_dropout0.3_l2`


In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)


In [ ]:
import os
import subprocess
from pathlib import Path

TARGET_FOLDER = Path('/content/drive/MyDrive/hallucination_detection')
REPO_URL = 'https://github.com/olgafilimonova2004/hallucination_detection_draft.git'
REPO_NAME = 'hallucination_detection_draft'
REPO_PATH = TARGET_FOLDER / REPO_NAME
AUTO_STASH = True

TARGET_FOLDER.mkdir(parents=True, exist_ok=True)

def run(cmd: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}: {cmd}')
    return result

print('TARGET_FOLDER =', TARGET_FOLDER)
print('REPO_PATH =', REPO_PATH)

if not REPO_PATH.exists():
    run(f'git clone {REPO_URL}', cwd=TARGET_FOLDER)

run('git remote -v', cwd=REPO_PATH)
run('git branch --show-current', cwd=REPO_PATH)
run('git fetch origin', cwd=REPO_PATH)
status = run('git status --short', cwd=REPO_PATH, check=False).stdout.strip()

if status:
    print('Local changes detected.')
    if AUTO_STASH:
        run('git stash push -u -m "colab-auto-stash"', cwd=REPO_PATH)
    else:
        raise RuntimeError('Repo is dirty. Set AUTO_STASH = True or clean it manually.')

run('git pull --ff-only origin main', cwd=REPO_PATH)
run('git log --oneline -1', cwd=REPO_PATH)

os.chdir(REPO_PATH)
print('cwd =', Path.cwd())


In [ ]:
run('pip install -q -r requirements.txt', cwd=REPO_PATH)


In [ ]:
import torch

print('torch.cuda.is_available() =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Switch Colab runtime to GPU.')


## Method 3 runs

Method 3 uses Qwen attentions directly, so the runner loads the model with eager attention internally.

The ablation runner writes one combined JSON file containing both requested classifier configurations.


In [ ]:
run(
    'python method3_llm_check/run_ablation.py '
    '--subset 40 '
    '--batch-size 1 '
    '--cache-dtype float32 '
    '--output-dir method3_llm_check/artifacts/ablation_smoke',
    cwd=REPO_PATH,
)


In [ ]:
run(
    'python method3_llm_check/run_ablation.py '
    '--batch-size 1 '
    '--cache-dtype float32',
    cwd=REPO_PATH,
)


In [ ]:
from pathlib import Path

results_path = REPO_PATH / 'method3_llm_check' / 'artifacts' / 'ablation' / 'ablation_results.json'
print(results_path)
print(results_path.read_text()[:12000])
